# Experimento de recortes: Docling vs visión directa

Notebook correspondiente a la **sección 4.3 de la memoria del TFM**, que documenta el experimento que justifica la estrategia de segmentación de imágenes adoptada en `imagen_pipeline.ipynb`.

## Pregunta de investigación

Cuando un estado financiero llega como imagen que comprime balance y PyG completos en un único plano visual, hay dos formas de extraer las partidas:

1. **Docling + LLM textual**: Docling extrae la estructura tabular mediante OCR y layout detection, y un LLM de texto identifica las partidas a partir del índice resultante.
2. **LLM multimodal de visión directa**: se pasa la imagen directamente a un modelo con capacidades de visión y se le pide que extraiga las partidas con sus valores.

El experimento evalúa ambos enfoques aplicando 0, 2, 3 y 4 recortes horizontales sobre la misma imagen, para determinar cuál es más robusto y cuál es el número óptimo de segmentos.

## Imagen de prueba

**Grifols, S.A.** — exportación directa del Excel de SABI a PNG que comprime balance consolidado y cuenta de PyG de tres ejercicios (2022, 2023, 2024) en una sola imagen. Se trata de un caso extremo de densidad: decenas de epígrafes jerárquicos, tres columnas de valores numéricos y cabeceras de metadatos coexisten en un único plano visual comprimido.

Se trabaja con dos versiones de la misma imagen:
- **Imagen densa** (`excel_balance.png`): exportación directa con la máxima densidad de información.
- **Imagen limpia** (`excel_balance (limpio).png`): misma imagen con mayor espaciado entre filas.

## Herramientas evaluadas

| Herramienta | Descripción |
|---|---|
| **Docling** (IBM) | Extracción estructurada de tablas con OCR (RapidOCR) y detección de layout. |
| **llama-4-scout-17b** (Meta/Groq) | Modelo multimodal con capacidades de visión, accesible gratis vía API de Groq. |

> Los modelos de mayor capacidad de OpenAI o Anthropic resuelven la extracción de visión directa sin problemas, pero el experimento busca una solución viable sin dependencia de modelos de pago. `llama-4-scout` es el modelo de visión gratuito más capaz disponible en Groq en el momento del experimento.

---


---

## Parte 1 — Docling con distinto número de recortes

### Hipótesis

Docling usa un modelo de detección de layout que necesita una separación visual mínima entre filas para identificarlas como entidades independientes. Cuando la imagen es muy densa, esa separación es inferior al umbral del modelo y varias filas adyacentes quedan fusionadas en una sola celda del DataFrame resultante.

**Si se segmenta la imagen en trozos horizontales antes de pasarla a Docling, cada trozo tiene menos filas y la separación relativa entre ellas aumenta. A más recortes, mejor debería ser la extracción.**

Se evalúan 4 configuraciones sobre la misma imagen: sin recortes (imagen completa), 2 recortes (mitades), 3 recortes (tercios) y 4 recortes (cuartos). En todas las configuraciones se añade un solapamiento del 5% en cada extremo del trozo para evitar que filas en la zona de corte queden partidas entre dos fragmentos.


### Imagen 1 — Exportación con máxima densidad

Primera prueba sobre la imagen más densa: balance y PyG comprimidos al máximo, columnas muy estrechas, separación mínima entre filas.


#### Celda 1 — Setup: carga de librerías, función de recorte y ejecución

Define `recortar_y_escalar`: recorta la imagen entre `top_pct` y `bottom_pct` (con overlap del 5% en cada extremo), escala el resultado ×2 con interpolación `LANCZOS` para mejorar la resolución que recibe RapidOCR, y guarda el trozo en disco.

Define `procesar_y_mostrar`: pasa cada trozo por `DocumentConverter` de Docling y muestra la tabla resultante para inspección visual.

> **Output:** confirmación de que la imagen se ha cargado correctamente y la ruta al fichero.


In [2]:
# ── CELDA 1: configuración ────────────────────────────────────────────────────
from PIL import Image
import pandas as pd
import os
from IPython.display import display
from docling.document_converter import DocumentConverter

# ► CAMBIA ESTA RUTA por la imagen que quieras analizar
RUTA_IMAGEN = r"C:\Users\perdi\Desktop\tfm\TFM PYTHON\RECORTES\excel_balance.png"

OUTPUTS = r"C:\Users\perdi\Desktop\tfm\TFM PYTHON\RECORTES\output recortes"
os.makedirs(OUTPUTS, exist_ok=True)

def recortar_y_escalar(ruta_imagen, top_pct, bottom_pct, escala=2, overlap_pct=0.05):
    img = Image.open(ruta_imagen)
    w, h = img.size
    top    = max(0, int(h * (top_pct - overlap_pct)))
    bottom = min(h, int(h * (bottom_pct + overlap_pct)))
    recorte = img.crop((0, top, w, bottom))
    recorte = recorte.resize((w * escala, (bottom - top) * escala), Image.LANCZOS)
    nombre = os.path.splitext(os.path.basename(ruta_imagen))[0]
    ruta_salida = os.path.join(OUTPUTS, f"{nombre}_exp_trozo_{top_pct}_{bottom_pct}.png")
    recorte.save(ruta_salida)
    return ruta_salida

def procesar_y_mostrar(trozos, etiqueta):
    converter = DocumentConverter()
    print(f"\n{'='*60}")
    print(f"  {etiqueta}")
    print(f"{'='*60}")
    for i, trozo in enumerate(trozos, 1):
        print(f"\n── Trozo {i} ──")
        result = converter.convert(trozo)
        n_tablas = len(result.document.tables)
        print(f"  Tablas detectadas: {n_tablas}")
        if n_tablas == 0:
            print("  (ninguna tabla encontrada)")
        for j, tabla in enumerate(result.document.tables, 1):
            print(f"  · Tabla {j}:")
            df_trozo = tabla.export_to_dataframe()
            df_trozo = pd.DataFrame(
                [df_trozo.columns.tolist()] + df_trozo.values.tolist()
            )
            df_trozo.columns = [f"C{k}" for k in range(df_trozo.shape[1])]
            display(df_trozo)

print("✅ Setup listo. Imagen:", RUTA_IMAGEN)

c:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Setup listo. Imagen: C:\Users\perdi\Desktop\tfm\TFM PYTHON\RECORTES\excel_balance.png


#### Celda 2 — Experimento: 0, 2, 3 y 4 recortes sobre la imagen densa

Ejecuta el mismo pipeline de Docling con cuatro granularidades distintas y muestra los DataFrames resultantes para cada configuración. Los outputs permiten comparar visualmente cómo evoluciona la calidad de extracción.

**Qué observar en los outputs:**
- **0 recortes (imagen completa):** múltiples filas fusionadas en una misma celda — una celda contiene el texto de dos, tres o más partidas concatenadas. Las columnas de valores también presentan errores de asignación.
- **2 recortes:** mejora notable. Persisten algunas fusiones en las zonas de corte superior e inferior de cada mitad, donde la densidad sigue siendo alta.
- **3 recortes:** la mayoría de partidas aparecen individualizadas. Las fusiones residuales se concentran en los bordes de los trozos.
- **4 recortes:** extracción limpia. Todas las partidas en sus propias filas, valores correctamente asignados a sus columnas de fecha.


In [3]:
# ── CELDA 2: experimento de recortes ─────────────────────────────────────────
# Cada bloque pasa la misma imagen por Docling con distinto número de cortes
# para evidenciar cómo más cortes => docling junta menos filas.

nombre = os.path.splitext(os.path.basename(RUTA_IMAGEN))[0]

# ── 0 recortes: imagen completa sin partir ────────────────────────────────────
img_completa = Image.open(RUTA_IMAGEN)
w, h = img_completa.size
img_escalada = img_completa.resize((w * 2, h * 2), Image.LANCZOS)
ruta_completa = os.path.join(OUTPUTS, f"{nombre}_exp_0recortes.png")
img_escalada.save(ruta_completa)
procesar_y_mostrar([ruta_completa], "SIN RECORTES (0 cortes) — imagen completa")

# ── 2 recortes: mitad superior + mitad inferior ───────────────────────────────
cortes_2 = [(0, 0.50), (0.50, 1.0)]
trozos_2 = [recortar_y_escalar(RUTA_IMAGEN, t, b) for t, b in cortes_2]
procesar_y_mostrar(trozos_2, "2 RECORTES — mitad superior / mitad inferior")

# ── 3 recortes: tres tercios ──────────────────────────────────────────────────
cortes_3 = [(0, 0.33), (0.33, 0.66), (0.66, 1.0)]
trozos_3 = [recortar_y_escalar(RUTA_IMAGEN, t, b) for t, b in cortes_3]
procesar_y_mostrar(trozos_3, "3 RECORTES — tres tercios iguales")

# ── 4 recortes: cuatro cuartos ────────────────────────────────────────────────
cortes_4 = [(0, 0.25), (0.25, 0.50), (0.50, 0.75), (0.75, 1.0)]
trozos_4 = [recortar_y_escalar(RUTA_IMAGEN, t, b) for t, b in cortes_4]
procesar_y_mostrar(trozos_4, "4 RECORTES — cuatro cuartos iguales")


  SIN RECORTES (0 cortes) — imagen completa

── Trozo 1 ──


[INFO] 2026-05-07 21:39:35,983 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-07 21:39:35,993 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-07 21:39:35,993 [RapidOCR] main.py:57: Using C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-07 21:39:36,123 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-07 21:39:36,126 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-07 21:39:36,126 [RapidOCR] main.py:57: Using C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-07 21:39:36,176 [RapidOCR] base.py:22: Using engine_nam

  Tablas detectadas: 1
  · Tabla 1:


,C0,C1,C2,C3,C4,C5,C6
0,ROW,CO C1 Balance/Estado de resultad nan,C2 nan,nan,S nan,C6 nan,C7 nan
1,3 4,Cuentas No Consolidadas nan,nan 31/12/2024 mil EUR,nan,31/12/2023,nan,31/12/2022
2,nan nan nan,nan nan,nan nan 12 meses nan,nan 12 meses,mil EUR nan,nan 12 meses,mil EUR nan
3,5 9 7,nan,Aprobado nan Normal PGC 2007 nan,Aprobado Normal PGC 2007,nan nan,Aprobado Normal PGC 2007,nan nan
4,8,nan nan Activo nan,nan nan,nan,nan,nan,nan
...,...,...,...,...,...,...,...
67,115 116,1. Importe neto de la cifr nan,nan 701053 nan,619242,nan,488639,nan nan
68,117,a) Ventas nan,n.d. nan,n.d.,nan,n.d.,nan
69,118,b) Prestaciones de servic nan 2. Variacion de ...,247819 nan,242824 n.d. 2312,nan nan nan,199311 n.d.,nan nan
70,119,,nan,,,5478,


[INFO] 2026-05-07 21:40:10,505 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-07 21:40:10,509 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-07 21:40:10,510 [RapidOCR] main.py:57: Using C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx


...

,C0,C1,C2,C3,C4,C5,C6,C7,C8
0,51,1Capital,nan,119604,nan,119604,nan,119604,nan
1,52,1. Capital escriturado,nan,119604,nan,119604,nan,119604,nan
2,53,2. (Capital no exigido),nan,n.d.,nan,n.d.,nan,n.d.,nan
3,54,I Prima de emision,nan,910728,nan,910728,nan,910728,nan
4,55,1. Reserva de revalorizad,nan,n.d.,nan,n.d.,nan,n.d.,nan
5,56,2.Reserva de capitalizac,nan,n.d.,nan,n.d.,nan,n.d.,nan
6,57,3.Otras reservas,nan,n.d.,nan,n.d.,nan,n.d.,nan
7,58,Il Reservas y resultadosd,nan,n.d.,nan,n.d.,nan,n.d.,nan
8,59,IV (Acciones y participacid,nan,-134448,nan,-152748,nan,-162220,nan
9,60,V Otras aportacionesde,nan,n.d.,nan,n.d.,nan,n.d.,nan



── Trozo 4 ──


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


  Tablas detectadas: 1
  · Tabla 1:


,C0,C1,C2,C3,C4,C5,C6,C7,C8
0,0,1,2,3,4,5,6,7,8
1,94,mDeudasacortoplazo,nan,68347,nan,106970,nan,74786,nan
2,95,1.obligacionesy otrosv,nan,38907,nan,30170,nan,12554,nan
3,96,2. Deudas con entidades,nan,29383,nan,68542,nan,60899,nan
4,97,3. Acreedores por arrend,nan,n.d.,nan,25,nan,391,nan
5,98,4. Otros pasivos financier,nan,n.d.,nan,n.d.,nan,n.d.,nan
6,99,IV Deudas con empresas,nan,62437,nan,64699,nan,61720,nan
7,100,1. Deudas con sociedade,nan,n.d.,nan,n.d.,nan,n.d.,nan
8,101,2. Otras deudas,nan,n.d.,nan,n.d.,nan,n.d.,nan
9,102,VAcreedorescomerciales,nan,94624,nan,112436,nan,82987,nan


#### Conclusión — Imagen densa

> Los outputs de la celda anterior confirman la hipótesis: **a mayor número de recortes, menor densidad de filas por trozo y mejor calidad de extracción con Docling**.
>
> El solapamiento del 5% entre trozos garantiza que ninguna partida quede perdida en la zona de corte, a costa de que algunas aparezcan duplicadas en dos trozos consecutivos — duplicidad fácilmente gestionable en la fase de construcción del DataFrame final.
>
> Con 4 recortes la imagen densa queda completamente legible para Docling. Este es el número adoptado en `imagen_pipeline.ipynb` para este tipo de imagen.


### Imagen 2 — Exportación con mayor espaciado

Segunda prueba con la versión más limpia de la imagen, que tiene mayor separación entre filas. Permite verificar que la mejora progresiva con más recortes se mantiene con imágenes de mejor calidad y comprobar si el número óptimo de recortes cambia.


#### Celda 3 — Setup con imagen limpia

Mismo setup que la celda 1, apuntando a `excel_balance (limpio).png`.


In [4]:
# ── CELDA 1: configuración ────────────────────────────────────────────────────
from PIL import Image
import pandas as pd
import os
from IPython.display import display
from docling.document_converter import DocumentConverter

# ► CAMBIA ESTA RUTA por la imagen que quieras analizar
RUTA_IMAGEN = r"C:\Users\perdi\Desktop\tfm\TFM PYTHON\RECORTES\excel_balance (limpio).png"

OUTPUTS = r"C:\Users\perdi\Desktop\tfm\TFM PYTHON\RECORTES\output recortes"
os.makedirs(OUTPUTS, exist_ok=True)

def recortar_y_escalar(ruta_imagen, top_pct, bottom_pct, escala=2, overlap_pct=0.05):
    img = Image.open(ruta_imagen)
    w, h = img.size
    top    = max(0, int(h * (top_pct - overlap_pct)))
    bottom = min(h, int(h * (bottom_pct + overlap_pct)))
    recorte = img.crop((0, top, w, bottom))
    recorte = recorte.resize((w * escala, (bottom - top) * escala), Image.LANCZOS)
    nombre = os.path.splitext(os.path.basename(ruta_imagen))[0]
    ruta_salida = os.path.join(OUTPUTS, f"{nombre}_exp_trozo_{top_pct}_{bottom_pct}.png")
    recorte.save(ruta_salida)
    return ruta_salida

def procesar_y_mostrar(trozos, etiqueta):
    converter = DocumentConverter()
    print(f"\n{'='*60}")
    print(f"  {etiqueta}")
    print(f"{'='*60}")
    for i, trozo in enumerate(trozos, 1):
        print(f"\n── Trozo {i} ──")
        result = converter.convert(trozo)
        n_tablas = len(result.document.tables)
        print(f"  Tablas detectadas: {n_tablas}")
        if n_tablas == 0:
            print("  (ninguna tabla encontrada)")
        for j, tabla in enumerate(result.document.tables, 1):
            print(f"  · Tabla {j}:")
            df_trozo = tabla.export_to_dataframe()
            df_trozo = pd.DataFrame(
                [df_trozo.columns.tolist()] + df_trozo.values.tolist()
            )
            df_trozo.columns = [f"C{k}" for k in range(df_trozo.shape[1])]
            display(df_trozo)

print("✅ Setup listo. Imagen:", RUTA_IMAGEN)

✅ Setup listo. Imagen: C:\Users\perdi\Desktop\tfm\TFM PYTHON\RECORTES\excel_balance (limpio).png


#### Celda 4 — Experimento: 0, 2, 3 y 4 recortes sobre la imagen limpia

Misma batería de configuraciones que en la imagen densa. Con mayor espaciado entre filas, Docling ya lo hace mejor desde 0 recortes, pero la mejora progresiva sigue siendo visible y 4 recortes sigue siendo el punto de máxima limpieza.

**Qué observar respecto a la imagen densa:**
- El punto de partida (0 recortes) es mejor: menos fusiones y valores más consistentes.
- La curva de mejora es más suave: ya con 2 recortes el resultado es usable.
- Con 4 recortes el resultado es idéntico al de la imagen densa con 4 recortes, confirmando que la estrategia converge independientemente de la calidad inicial de la imagen.


In [5]:
# ── CELDA 2: experimento de recortes ─────────────────────────────────────────
# Cada bloque pasa la misma imagen por Docling con distinto número de cortes
# para evidenciar cómo más cortes => docling junta menos filas.

nombre = os.path.splitext(os.path.basename(RUTA_IMAGEN))[0]

# ── 0 recortes: imagen completa sin partir ────────────────────────────────────
img_completa = Image.open(RUTA_IMAGEN)
w, h = img_completa.size
img_escalada = img_completa.resize((w * 2, h * 2), Image.LANCZOS)
ruta_completa = os.path.join(OUTPUTS, f"{nombre}_exp_0recortes.png")
img_escalada.save(ruta_completa)
procesar_y_mostrar([ruta_completa], "SIN RECORTES (0 cortes) — imagen completa")

# ── 2 recortes: mitad superior + mitad inferior ───────────────────────────────
cortes_2 = [(0, 0.50), (0.50, 1.0)]
trozos_2 = [recortar_y_escalar(RUTA_IMAGEN, t, b) for t, b in cortes_2]
procesar_y_mostrar(trozos_2, "2 RECORTES — mitad superior / mitad inferior")

# ── 3 recortes: tres tercios ──────────────────────────────────────────────────
cortes_3 = [(0, 0.33), (0.33, 0.66), (0.66, 1.0)]
trozos_3 = [recortar_y_escalar(RUTA_IMAGEN, t, b) for t, b in cortes_3]
procesar_y_mostrar(trozos_3, "3 RECORTES — tres tercios iguales")

# ── 4 recortes: cuatro cuartos ────────────────────────────────────────────────
cortes_4 = [(0, 0.25), (0.25, 0.50), (0.50, 0.75), (0.75, 1.0)]
trozos_4 = [recortar_y_escalar(RUTA_IMAGEN, t, b) for t, b in cortes_4]
procesar_y_mostrar(trozos_4, "4 RECORTES — cuatro cuartos iguales")

[INFO] 2026-05-07 21:46:51,322 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-07 21:46:51,326 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-07 21:46:51,327 [RapidOCR] main.py:57: Using C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx



  SIN RECORTES (0 cortes) — imagen completa

── Trozo 1 ──


[INFO] 2026-05-07 21:46:51,423 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-07 21:46:51,425 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-07 21:46:51,425 [RapidOCR] main.py:57: Using C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-07 21:46:51,472 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-07 21:46:51,482 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_mobile.onnx
[INFO] 2026-05-07 21:46:51,483 [RapidOCR] main.py:57: Using C:\Users\perdi\Desktop\tfm\TFM PYTHON\entornotfm\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_mobile.onnx
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 10438.89it/s]


  Tablas detectadas: 1
  · Tabla 1:


,C0,C1,C2,C3,C4,C5,C6
0,Balance/Estado de resultados.Cuentas No Consol...,31/12/2024,31/12/2023,,,,
1,,mil EUR,12 meses,,,,
2,Activo,Aprobado Normal PGC 2007,Aprobado Normal PGC 2007,Aprobado Normal PGC 2007,,,
3,A) Activo no corriente,12497590,11416223,12647002,,,
4,1 Inmovilizado intangible 1. Fondo de comerci...,23959 n.d.,19941 n.d.,23213 n.d.,,,
...,...,...,...,...,...,...,...
69,,,,n.d. 12893101,Vll Deuda con caracteristicas especiales a cor...,,
70,,,,n.d. n.d.,,,
71,,,,,,,C) Pasivo corriente 1. Obligaciones y otros va...
72,,,,n.d.,,,


...

,C0,C1,C2,C3
0,A-1)Fondospropios,1972243,2044735,2285248
1,ICapital,119604,119604,119604
2,1.Capital escriturado,119604,119604,119604
3,2. (Capital no exigido),n.d.,n.d.,n.d.
4,IlPrima de emision,910728,910728,910728
5,1.Reserva derevalorizacion,n.d.,n.d.,n.d.
6,2.Reserva de capitalizacion,n.d.,n.d.,n.d.
7,3.Otrasreservas,n.d.,n.d.,n.d.
8,Il Reservasyresultadosdeejerciciosanteriores,n.d.,n.d.,n.d.
9,IV(Accionesyparticipacionesenpatrimoniopropias...,-134448,-152748,-162220



── Trozo 4 ──


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


  Tablas detectadas: 1
  · Tabla 1:


,C0,C1,C2,C3
0,3.Acreedoresporarrendamientofinanciero,,21 27,52
1,4. Otros pasivos financieros,n.d.,n.d.,n.d.
2,IlI Deudasconempresasdel grupoyasociadas alarg...,4276854,4673555,6419171
3,1.Deudasconsociedadespuestaenequivalencia,n.d.,n.d.,n.d.
4,2.Otras deudas,n.d.,n.d.,n.d.
5,IVPasivosporimpuestodiferido,3244,4907,2580
6,VPeriodificacionesalargoplazo,n.d.,n.d.,n.d.
7,VIAcreedorescomercialesnocorrientes,n.d.,n.d.,n.d.
8,VllDeudaconcaracteristicasespecialesalargoplazo,n.d.,n.d.,n.d.
9,C)Pasivo corriente,313933,298105,226493


#### Conclusión — Imagen limpia

> La mejora progresiva con más recortes se confirma también en imágenes de mejor calidad. El número óptimo de recortes **no depende de la calidad de la imagen sino de la densidad de filas**: una imagen de alta calidad con muchas filas necesita tantos recortes como una de baja calidad con la misma densidad.


---

## Parte 2 — LLM multimodal de visión directa

### Motivación

El enfoque alternativo más intuitivo es enviar la imagen directamente a un modelo con capacidades de visión y pedirle que extraiga las 22 partidas con sus valores en un único paso, sin necesidad de Docling ni de serialización textual.

Este enfoque tiene ventajas teóricas: el modelo ve la imagen tal como la vería un humano, puede interpretar el contexto visual completo y no depende de la calidad del OCR de Docling. La pregunta es si un modelo de visión gratuito (`llama-4-scout-17b`) es suficientemente fiable para esta tarea con imágenes de alta densidad.

### Hipótesis inversa

Al contrario que con Docling, **segmentar la imagen podría perjudicar al modelo de visión**: cuando se corta la imagen, los trozos inferiores pierden la cabecera con las fechas de los ejercicios, y el modelo ya no tiene referencia para saber a qué año corresponde cada columna de valores.


#### Celda 5 — Setup: recortes y cliente Groq para visión

Genera los mismos recortes que en la Parte 1 (0, 2, 3 y 4 fragmentos) y configura el cliente Groq con `llama-4-scout-17b-16e-instruct`, el modelo de visión disponible en el plan gratuito.


In [1]:
# celda 1



from PIL import Image
import os

RUTA_IMAGEN = r"C:\Users\david\Desktop\david\tfm\TFM PYTHON\RECORTES EXPERIMENTO\excel_balance.png"
OUTPUTS = r"C:\Users\david\Desktop\david\tfm\TFM PYTHON\outputs"
os.makedirs(OUTPUTS, exist_ok=True)

def recortar_y_escalar(ruta_imagen, top_pct, bottom_pct, escala=2, overlap_pct=0.05):
    img = Image.open(ruta_imagen)
    w, h = img.size
    top    = max(0, int(h * (top_pct - overlap_pct)))
    bottom = min(h, int(h * (bottom_pct + overlap_pct)))
    recorte = img.crop((0, top, w, bottom))
    recorte = recorte.resize((w * escala, (bottom - top) * escala), Image.LANCZOS)
    nombre = os.path.splitext(os.path.basename(ruta_imagen))[0]
    ruta_salida = os.path.join(OUTPUTS, f"{nombre}_exp_trozo_{top_pct}_{bottom_pct}.png")
    recorte.save(ruta_salida)
    return ruta_salida

# 0 recortes
img = Image.open(RUTA_IMAGEN)
w, h = img.size
img_escalada = img.resize((w * 2, h * 2), Image.LANCZOS)
nombre = os.path.splitext(os.path.basename(RUTA_IMAGEN))[0]
ruta_completa = os.path.join(OUTPUTS, f"{nombre}_exp_0recortes.png")
img_escalada.save(ruta_completa)

# 2 recortes
trozos_2 = [recortar_y_escalar(RUTA_IMAGEN, t, b) for t, b in [(0, 0.50), (0.50, 1.0)]]

# 3 recortes
trozos_3 = [recortar_y_escalar(RUTA_IMAGEN, t, b) for t, b in [(0, 0.33), (0.33, 0.66), (0.66, 1.0)]]

# 4 recortes
trozos_4 = [recortar_y_escalar(RUTA_IMAGEN, t, b) for t, b in [(0, 0.25), (0.25, 0.50), (0.50, 0.75), (0.75, 1.0)]]

print("✅ Recortes listos:")
print(f"  0 recortes: {ruta_completa}")
print(f"  2 recortes: {trozos_2}")
print(f"  3 recortes: {trozos_3}")
print(f"  4 recortes: {trozos_4}")

✅ Recortes listos:
  0 recortes: C:\Users\david\Desktop\david\tfm\TFM PYTHON\outputs\excel_balance_exp_0recortes.png
  2 recortes: ['C:\\Users\\david\\Desktop\\david\\tfm\\TFM PYTHON\\outputs\\excel_balance_exp_trozo_0_0.5.png', 'C:\\Users\\david\\Desktop\\david\\tfm\\TFM PYTHON\\outputs\\excel_balance_exp_trozo_0.5_1.0.png']
  3 recortes: ['C:\\Users\\david\\Desktop\\david\\tfm\\TFM PYTHON\\outputs\\excel_balance_exp_trozo_0_0.33.png', 'C:\\Users\\david\\Desktop\\david\\tfm\\TFM PYTHON\\outputs\\excel_balance_exp_trozo_0.33_0.66.png', 'C:\\Users\\david\\Desktop\\david\\tfm\\TFM PYTHON\\outputs\\excel_balance_exp_trozo_0.66_1.0.png']
  4 recortes: ['C:\\Users\\david\\Desktop\\david\\tfm\\TFM PYTHON\\outputs\\excel_balance_exp_trozo_0_0.25.png', 'C:\\Users\\david\\Desktop\\david\\tfm\\TFM PYTHON\\outputs\\excel_balance_exp_trozo_0.25_0.5.png', 'C:\\Users\\david\\Desktop\\david\\tfm\\TFM PYTHON\\outputs\\excel_balance_exp_trozo_0.5_0.75.png', 'C:\\Users\\david\\Desktop\\david\\tfm\\TFM P

#### Celda 6 — Inicialización del cliente de visión

Carga la API key desde `.env` y confirma el modelo de visión activo. Los tres clientes en rotación se usan para distribuir la carga entre las múltiples llamadas del experimento y respetar los límites RPM del plan gratuito de Groq.


In [2]:
# CELDA 2

import os
from dotenv import load_dotenv
import openai
import json

load_dotenv()

GROQ_BASE_URL = "https://api.groq.com/openai/v1"

clients = [
    openai.OpenAI(api_key=os.getenv("GROQ_API_KEY_1", ""), base_url=GROQ_BASE_URL),
    openai.OpenAI(api_key=os.getenv("GROQ_API_KEY_2", ""), base_url=GROQ_BASE_URL),
    openai.OpenAI(api_key=os.getenv("GROQ_API_KEY_3", ""), base_url=GROQ_BASE_URL),
]

MODELO_VISION = "meta-llama/llama-4-scout-17b-16e-instruct"

print("✅ Cliente Groq listo. Modelo visión:", MODELO_VISION)

✅ Cliente Groq listo. Modelo visión: meta-llama/llama-4-scout-17b-16e-instruct


#### Celda 7 — Setup de partidas objetivo y función base64

Define la lista de 22 alias a extraer y la función `imagen_a_base64` que codifica la imagen para enviarla al modelo multimodal.


In [3]:
# CELDA 3 - SETUP VISION
import base64

PARTIDAS_OBJETIVO = [
    "patrimonio_neto", "fondos_propios", "pasivo_no_corriente",
    "deudas_lp", "deudas_cp", "pasivo_corriente", "total_activo",
    "activo_corriente", "existencias", "deudores_comerciales", "efectivo",
    "inmovilizado_material", "acreedores_comerciales", "cifra_negocios",
    "otros_ingresos_explotacion", "resultado_explotacion", "amortizacion",
    "gastos_financieros", "resultado_antes_impuestos", "resultado_ejercicio",
    "deuda_credito_lp", "deuda_credito_cp"
]

def imagen_a_base64(ruta):
    with open(ruta, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

print("✅ Setup listo")

✅ Setup listo


#### Celda 8 — Visión directa: imagen completa, prompt inicial

Primera llamada: imagen completa sin recortar, prompt básico que lista las 22 partidas y pide un JSON.

**Qué observar en el output:**
- El modelo devuelve las 22 partidas (`finish_reason: stop`, no trunca).
- Sin embargo los valores son **incorrectos**: `pasivo_no_corriente` aparece con valor `90.567.251` cuando el real es `10.572.251` (un dígito de más). El `resultado_antes_impuestos` recibe el valor del total del activo. Varios epígrafes tienen solo 2 valores en lugar de 3 (ignora un ejercicio).
- La PyG aparece sin valores o con valores inventados.

El modelo ve la imagen pero no la lee con precisión suficiente para valores numéricos en tablas densas.


In [4]:
# CELDA 4 - DEBUG RAW (sin parseo, ves exactamente lo que devuelve Groq)

def debug_raw(ruta_imagen, etiqueta):
    b64 = imagen_a_base64(ruta_imagen)
    prompt = f"""Eres un extractor de datos financieros. Esta imagen es un fragmento ({etiqueta}) de un balance/cuenta de pérdidas y ganancias.

Lista de partidas a extraer:
{', '.join(PARTIDAS_OBJETIVO)}

Devuelve SOLO JSON sin markdown con las partidas que encuentres:
{{
  "nombre_partida": {{"partida": "texto literal", "valores": [val1, val2]}},
  ...
}}"""

    response = clients[0].chat.completions.create(
        model=MODELO_VISION,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": prompt}
            ]
        }],
        temperature=0,
        max_tokens=2048
    )

    finish_reason = response.choices[0].finish_reason
    tokens_in  = response.usage.prompt_tokens
    tokens_out = response.usage.completion_tokens
    raw = response.choices[0].message.content

    print(f"finish_reason : {finish_reason}")
    print(f"tokens        : {tokens_in} in / {tokens_out} out")
    print(f"--- RAW ---")
    print(raw)
    print(f"--- FIN ---\n")

# Prueba primero solo imagen completa
debug_raw(ruta_completa, "trozo 1 de 1")

finish_reason : stop
tokens        : 2526 in / 694 out
--- RAW ---
```json
{
  "patrimonio_neto": {"partida": "Patrimonio Neto", "valores": [1953852, 2343325]},
  "fondos_propios": {"partida": "A) Fondos propios", "valores": [1972243, 2285248]},
  "pasivo_no_corriente": {"partida": "Pasivo No Corriente", "valores": [90567251, 90906126]},
  "deudas_lp": {"partida": "Deudas a largo plazo", "valores": [6288936, 5874925]},
  "deudas_cp": {"partida": "Deudas a corto plazo", "valores": [79947, 106970]},
  "pasivo_corriente": {"partida": "Pasivo Corriente", "valores": [313933, 298105]},
  "total_activo": {"partida": "Total Activo", "valores": [12840036, 12961217]},
  "activo_corriente": {"partida": "Activo Corriente", "valores": [290309, 324958]},
  "existencias": {"partida": "Existencias", "valores": [13342, 11359]},
  "deudores_comerciales": {"partida": "Deudores Comerciales", "valores": [79885, 72869]},
  "efectivo": {"partida": "Efectivo", "valores": [1283, 13678]},
  "inmovilizado_materi

#### Celda 9 — Visión directa: imagen completa, prompt mejorado

Segunda iteración con reglas más estrictas: solo incluir partidas vistas literalmente, un valor por columna de fecha, no inventar datos. Se añade extracción del bloque JSON con regex para aislar el resultado aunque haya texto adicional.

**Qué observar en el output:**
- Con instrucciones más estrictas el modelo omite más partidas (no inventa) pero los valores siguen siendo incorrectos donde los hay.
- El problema no es el prompt sino la limitación estructural del modelo para leer columnas de datos numéricos en imágenes de alta densidad.


In [8]:
# CELDA 4 - DEBUG RAW PROMPT MEJORADO

import re

def debug_raw(ruta_imagen, etiqueta):
    b64 = imagen_a_base64(ruta_imagen)
    prompt = f"""Eres un extractor de datos financieros experto en balances y cuentas de pérdidas y ganancias españoles.

Esta imagen es un fragmento ({etiqueta}) de un estado financiero.

PARTIDAS QUE BUSCO:
{', '.join(PARTIDAS_OBJETIVO)}

REGLAS ESTRICTAS:
1. Solo incluye en el JSON una partida si la ves LITERALMENTE en la imagen con su texto y sus valores numéricos.
2. Si no encuentras una partida, NO la incluyas. Ni con null, ni con [], ni con nan. Simplemente no la pongas.
3. Para cada partida encontrada, devuelve UN valor por cada columna de fecha/ejercicio que veas. Si hay 3 fechas, 3 valores. Si hay 2, 2 valores. En el mismo orden izquierda a derecha.
4. Los valores son siempre números enteros o decimales. Si en la imagen pone n.d. o está vacío, no incluyas esa partida.
5. No confundas partidas: cada clave del JSON debe corresponder exactamente a la partida que describes en "partida".
6. No inventes valores. Si no lees claramente un número, no lo incluyas.

Devuelve SOLO el JSON, sin texto antes ni después, sin markdown, sin explicaciones:
{{
  "nombre_partida": {{"partida": "texto literal exacto de la imagen", "valores": [val1, val2, val3]}},
  ...
}}"""

    response = clients[0].chat.completions.create(
        model=MODELO_VISION,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": prompt}
            ]
        }],
        temperature=0,
        max_tokens=2048
    )

    finish_reason = response.choices[0].finish_reason
    tokens_in  = response.usage.prompt_tokens
    tokens_out = response.usage.completion_tokens
    raw = response.choices[0].message.content

    # Extraer solo el bloque JSON aunque haya texto antes o después
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    raw_limpio = match.group(0) if match else raw

    print(f"finish_reason : {finish_reason}")
    print(f"tokens        : {tokens_in} in / {tokens_out} out")
    print(f"--- RAW ---")
    print(raw)
    print(f"--- FIN ---\n")

    return raw_limpio

# Prueba con imagen completa
debug_raw(ruta_completa, "trozo 1 de 1")

finish_reason : stop
tokens        : 2737 in / 674 out
--- RAW ---
{
  "patrimonio_neto": {"partida": "Patrimonio neto", "valores": [1953852]},
  "fondos_propios": {"partida": "A-1) Fondos propios", "valores": [1972243, 2044735]},
  "pasivo_no_corriente": {"partida": "Pasivo no corriente", "valores": [90572251, 9039245]},
  "deudas_lp": {"partida": "Deudas a largo plazo", "valores": [6288936, 5879925]},
  "deudas_cp": {"partida": "Deudas a corto plazo", "valores": [79947, 14000]},
  "pasivo_corriente": {"partida": "C) Pasivo corriente", "valores": [313933, 298105]},
  "total_activo": {"partida": "Total activo (A + B)", "valores": [128640036, 12961217]},
  "activo_corriente": {"partida": "B) Activo corriente", "valores": [2280981]},
  "existencias": {"partida": "Existencias", "valores": [13342, 12253]},
  "deudores_comerciales": {"partida": "Clientes por ventas y prestación de servicios", "valores": [1496, 581]},
  "efectivo": {"partida": "Efectivo y otros activos líquidos", "valores": 

'{\n  "patrimonio_neto": {"partida": "Patrimonio neto", "valores": [1953852]},\n  "fondos_propios": {"partida": "A-1) Fondos propios", "valores": [1972243, 2044735]},\n  "pasivo_no_corriente": {"partida": "Pasivo no corriente", "valores": [90572251, 9039245]},\n  "deudas_lp": {"partida": "Deudas a largo plazo", "valores": [6288936, 5879925]},\n  "deudas_cp": {"partida": "Deudas a corto plazo", "valores": [79947, 14000]},\n  "pasivo_corriente": {"partida": "C) Pasivo corriente", "valores": [313933, 298105]},\n  "total_activo": {"partida": "Total activo (A + B)", "valores": [128640036, 12961217]},\n  "activo_corriente": {"partida": "B) Activo corriente", "valores": [2280981]},\n  "existencias": {"partida": "Existencias", "valores": [13342, 12253]},\n  "deudores_comerciales": {"partida": "Clientes por ventas y prestación de servicios", "valores": [1496, 581]},\n  "efectivo": {"partida": "Efectivo y otros activos líquidos", "valores": [1283, 13678]},\n  "inmovilizado_material": {"partida":

#### Celda 10 — Visión directa: 2 recortes

Se segmenta la imagen en 2 mitades y se envía cada una al modelo por separado.

**Qué observar en el output:**
- El trozo superior (que contiene la cabecera con fechas) devuelve resultados parciales.
- El trozo inferior pierde la referencia temporal: el modelo ya no sabe a qué fecha corresponde cada columna y devuelve valores sin asignación correcta o JSON malformado.
- Solo 9 de las 22 partidas quedan correctamente identificadas entre los dos trozos.

La segmentación empeora el resultado con visión directa, confirmando la hipótesis inversa.


In [5]:
# CELDA 5 - DEBUG 2 RECORTES

import time
import re

def debug_raw(ruta_imagen, etiqueta):
    b64 = imagen_a_base64(ruta_imagen)
    prompt = f"""Eres un extractor de datos financieros. Esta imagen es un fragmento ({etiqueta}) de un balance/cuenta de pérdidas y ganancias.

Lista de partidas a extraer:
{', '.join(PARTIDAS_OBJETIVO)}

Devuelve SOLO JSON sin markdown con las partidas que encuentres:
{{
  "nombre_partida": {{"partida": "texto literal", "valores": [val1, val2]}},
  ...
}}"""

    response = clients[0].chat.completions.create(
        model=MODELO_VISION,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": prompt}
            ]
        }],
        temperature=0,
        max_tokens=2048
    )

    finish_reason = response.choices[0].finish_reason
    tokens_in  = response.usage.prompt_tokens
    tokens_out = response.usage.completion_tokens
    raw = response.choices[0].message.content
    raw_limpio = re.sub(r"```json\s*|\s*```", "", raw).strip()

    print(f"finish_reason : {finish_reason}")
    print(f"tokens        : {tokens_in} in / {tokens_out} out")
    print(f"--- RAW ---")
    print(raw)
    print(f"--- FIN ---\n")
    
    return raw_limpio

print("=== 2 RECORTES ===")
for i, ruta in enumerate(trozos_2, 1):
    print(f"\n── Trozo {i}/2 ──")
    print(f"ruta: {ruta}")
    debug_raw(ruta, f"trozo {i} de 2")
    if i < len(trozos_2):
        print("esperando 5s...")
        time.sleep(5)

=== 2 RECORTES ===

── Trozo 1/2 ──
ruta: C:\Users\david\Desktop\david\tfm\TFM PYTHON\outputs\excel_balance_exp_trozo_0_0.5.png
finish_reason : stop
tokens        : 2671 in / 658 out
--- RAW ---
```json
{
  "patrimonio_neto": {"partida": "AJ) Patrimonio neto", "valores": [1953852, 2101487, 2343125]},
  "fondos_propios": {"partida": "A) Fondos propios", "valores": [1972243, 2044735, 2285248]},
  "pasivo_no_corriente": {"partida": null, "valores": []},
  "deudas_lp": {"partida": "V. Deudas a largo plazo", "valores": [418896, 2714, 29199]},
  "deudas_cp": {"partida": "VII. Deudas comerciales no corrientes", "valores": [null, null, null]},
  "pasivo_corriente": {"partida": null, "valores": []},
  "total_activo": {"partida": "Total activo (A + B)", "valores": [12840036, 12961217, 12893101]},
  "activo_corriente": {"partida": "A) Activo no corriente", "valores": [12497590, 11416223, 12647002]},
  "existencias": {"partida": "I. Existencias", "valores": [13342, 12333, 11439]},
  "deudores_come

#### Celda 11 — Visión directa: 3 recortes

Misma prueba con 3 fragmentos.

**Qué observar:** el tercer trozo produce un JSON truncado (`finish_reason: length`) porque el output de 22 partidas para un fragmento pequeño supera el límite de tokens de salida. A medida que los trozos son más pequeños, el modelo tiene menos contexto y más dificultad para generar el JSON completo.


In [6]:
# CELDA 6 - DEBUG 3 RECORTES

print("=== 3 RECORTES ===")
for i, ruta in enumerate(trozos_3, 1):
    print(f"\n── Trozo {i}/3 ──")
    print(f"ruta: {ruta}")
    debug_raw(ruta, f"trozo {i} de 3")
    if i < len(trozos_3):
        print("esperando 5s...")
        time.sleep(5)

=== 3 RECORTES ===

── Trozo 1/3 ──
ruta: C:\Users\david\Desktop\david\tfm\TFM PYTHON\outputs\excel_balance_exp_trozo_0_0.33.png
finish_reason : stop
tokens        : 2526 in / 690 out
--- RAW ---
Lo siento, pero no puedo extraer datos directamente de la imagen proporcionada. Sin embargo, puedo guiarte sobre cómo abordar la extracción de datos financieros de un balance o cuenta de pérdidas y ganancias.

Para extraer las partidas solicitadas, necesitaría ver los valores específicos en el documento o tabla que estás analizando. Dado que no puedo ver la imagen directamente, te proporcionaré una estructura JSON con los campos que mencionaste, pero sin valores, ya que no puedo acceder a la información visual.

```json
{
  "patrimonio_neto": {"partida": " Patrimonio Neto", "valores": []},
  "fondos_propios": {"partida": "Fondos Propios", "valores": []},
  "pasivo_no_corriente": {"partida": "Pasivo No Corriente", "valores": []},
  "deudas_lp": {"partida": "Deudas a Largo Plazo", "valores": []}

#### Celda 12 — Visión directa: 4 recortes

Con 4 fragmentos el colapso es total: todos los trozos producen JSON truncado o vacío. El modelo no puede generar el JSON de 22 partidas para un fragmento que solo contiene una cuarta parte del balance.

**Conclusión de la batería:** el rendimiento de visión directa **empeora monotónicamente** al aumentar los recortes, en el patrón exactamente opuesto a Docling.


In [7]:
# CELDA 7 - DEBUG 4 RECORTES

print("=== 4 RECORTES ===")
for i, ruta in enumerate(trozos_4, 1):
    print(f"\n── Trozo {i}/4 ──")
    print(f"ruta: {ruta}")
    debug_raw(ruta, f"trozo {i} de 4")
    if i < len(trozos_4):
        print("esperando 5s...")
        time.sleep(5)

=== 4 RECORTES ===

── Trozo 1/4 ──
ruta: C:\Users\david\Desktop\david\tfm\TFM PYTHON\outputs\excel_balance_exp_trozo_0_0.25.png
finish_reason : stop
tokens        : 2526 in / 810 out
--- RAW ---
Lo siento, pero no puedo ver la imagen que mencionas. Sin embargo, puedo guiarte sobre cómo abordar la extracción de datos financieros de una tabla como la que describes. 

Basándome en la estructura que proporcionaste, aquí te dejo un ejemplo de cómo podría estructurarse el JSON con las partidas que mencionas. Ten en cuenta que no tengo los valores específicos ya que no puedo visualizar la imagen:

```json
{
  "patrimonio_neto": {"partida": " Patrimonio Neto", "valores": [nan, nan, nan]},
  "fondos_propios": {"partida": "Fondos Propios", "valores": [nan, nan, nan]},
  "pasivo_no_corriente": {"partida": "Pasivo No Corriente", "valores": [nan, nan, nan]},
  "deudas_lp": {"partida": "Deudas a Largo Plazo", "valores": [nan, nan, nan]},
  "deudas_cp": {"partida": "Deudas a Corto Plazo", "valores":

---

### Resultados del experimento de visión

El experimento evalúa las 4 configuraciones de recorte (0, 2, 3, 4) con dos versiones del prompt: una inicial más directa y una mejorada con reglas más estrictas. Los outputs de las siguientes celdas muestran el JSON raw devuelto por el modelo para cada configuración.

**Tabla resumen de resultados (documentada en la memoria):**

| Recortes | Partidas encontradas | Tokens totales | Problemas observados |
|:---:|:---:|:---:|---|
| **0** (imagen completa) | 22/22 | 3.309 | Valores inventados, solo 2 valores por partida en vez de 3, PyG vacía |
| **2** | 9/22 | 6.333 | JSON truncado en trozo 1, pérdida de referencia temporal |
| **3** | 12/22 | 8.958 | JSON truncado en trozo 3, valores incoherentes |
| **4** | 0/22 | 12.269 | JSON truncado en todos los trozos |



---

## Conclusiones del experimento

### Docling con recortes ✅

La densidad de filas por imagen es el **factor determinante** de la calidad de extracción con Docling. El recorte en fragmentos horizontales con solapamiento es el mecanismo más efectivo para reducirla.

La mejora es progresiva y consistente tanto en imágenes densas como limpias:

| Recortes | Resultado con Docling |
|:---:|---|
| 0 | Filas fusionadas, errores estructurales graves |
| 2 | Mejora notable, fusiones residuales en bordes |
| 3 | Mayoría de partidas individualizadas |
| **4** | **Extracción limpia, resultado correcto** |

### Visión directa ❌

El enfoque de visión directa con `llama-4-scout` (modelo gratuito) **no es viable** para este tipo de documentos. Sus limitaciones son estructurales:

- Con imagen completa: identifica las 22 partidas pero con valores incorrectos, columnas mezcladas y PyG vacía.
- Al segmentar: los trozos pierden la cabecera con las fechas y el JSON se trunca por superar el límite de tokens de salida.
- El coste en tokens **escala linealmente** con el número de recortes sin ninguna mejora de calidad.

> **Nota importante:** modelos de mayor capacidad (GPT-4o, Claude Sonnet) resuelven la extracción por visión directa sin estos problemas. La limitación es específica de los modelos gratuitos disponibles en el momento del experimento. El pipeline Docling se adoptó para garantizar reproducibilidad sin coste de inferencia.

### Heurística de recortes adoptada

| Tipo de documento | Recortes | Justificación |
|---|:---:|---|
| Imagen única con balance+PyG completos | 4 | Alta densidad, todo comprimido en un plano |
| PDF — páginas con balance completo | 0 | Cada página tiene densidad baja, Docling lee bien directamente |
| PDF — páginas con continuación de tabla | 0 | Idem |

El número óptimo de recortes **no es universal**: depende de la densidad de filas de cada imagen concreta. En producción, podría automatizarse detectando celdas fusionadas en el output de Docling e incrementando el número de recortes hasta eliminarlas.
